In [6]:
import numpy as np
import mne
from scipy.io import loadmat
from scipy.signal import welch
import matplotlib.pyplot as plt
from MFDFA import MFDFA
import glob

In [7]:
# 1. Load and Preprocess Data
def load_bci_data(file_path):
    """Load a single .mat file and return BCI structure."""
    mat = loadmat(file_path)
    bci = mat['BCI']
    # Check if BCI is a nested array
    if isinstance(bci, np.ndarray) and bci.shape == (1, 1):
        bci = bci[0, 0]
    return bci

def create_mne_raw(bci, trial_idx=4):
    """Create MNE Raw object for a single trial."""
    # Get channel names
    try:
        # Check if bci is a structured array or dictionary
        if hasattr(bci, 'dtype') and bci.dtype.names and 'chaninfo' in bci.dtype.names:
            chaninfo = bci['chaninfo']
            if chaninfo.dtype.names and 'label' in chaninfo.dtype.names:
                label_field = chaninfo[0, 0]['label']
            else:
                raise KeyError("label field not found in chaninfo")
        elif isinstance(bci, dict) and 'chaninfo' in bci:
            chaninfo = bci['chaninfo']
            if isinstance(chaninfo, np.ndarray) and chaninfo.dtype.names and 'label' in chaninfo.dtype.names:
                label_field = chaninfo[0, 0]['label']
            elif isinstance(chaninfo, dict) and 'label' in chaninfo:
                label_field = chaninfo['label']
            else:
                raise KeyError("label field not found in chaninfo")
        else:
            raise KeyError("chaninfo field not found in BCI")
        
        # Convert label_field to list
        if isinstance(label_field, np.ndarray) and label_field.size > 0:
            ch_names = label_field.flatten().tolist()
        elif isinstance(label_field, (list, tuple)):
            ch_names = list(label_field)
        else:
            raise ValueError(f"Unexpected label field structure: {type(label_field)}")
    except (IndexError, KeyError, ValueError) as e:
        print(f"Error accessing chaninfo.label: {e}")
        print("BCI type:", type(bci))
        print("BCI keys or dtype.names:", bci.dtype.names if hasattr(bci, 'dtype') else list(bci.keys()) if isinstance(bci, dict) else "Unknown")
        raise
    
    # Get EEG data
    try:
        eeg_data = bci['data'][0, trial_idx][0]  # Shape: (nChannels, nTime)
    except (IndexError, KeyError):
        print(f"Error: trial_idx {trial_idx} is out of bounds or data field missing. Available trials: {bci['data'].shape[1] if 'data' in bci else 'Unknown'}")
        raise
    
    sfreq = bci['SRATE'][0, 0]  # 1000 Hz
    
    # Create MNE Info object
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    
    # Create Raw object
    raw = mne.io.RawArray(eeg_data * 1e-6, info)  # Convert μV to V for MNE
    raw.set_montage('standard_1020')  # Use standard 10-10 montage
    
    # Filter for alpha band (8-14 Hz)
    raw.filter(8, 14, fir_design='firwin')
    
    # Exclude noisy channels
    try:
        noisy_channels = bci['chaninfo'][0, 0]['noisechan'] if hasattr(bci, 'dtype') else bci['chaninfo']['noisechan']
        if len(noisy_channels) > 0:
            bad_channels = [ch_names[i] for i in noisy_channels]
            raw.info['bads'] = bad_channels
            raw.interpolate_bads()
    except (IndexError, KeyError):
        print("No noisy channels found or error accessing noisechan.")
    
    return raw

In [8]:
# 2. Visualizations
def plot_time_series(raw, channels=['C3', 'C4']):
    """Plot EEG time series for selected channels."""
    try:
        raw.plot(scalings='auto', n_channels=2, show=False, block=False, picks=channels)
        plt.title('EEG Time Series (C3, C4)')
        plt.savefig('eeg_time_series.png')
        plt.close()
    except ValueError as e:
        print(f"Error plotting time series: {e}. Check if channels {channels} exist.")

def plot_psd(raw, channels=['C3', 'C4']):
    """Plot PSD for selected channels."""
    try:
        raw.compute_psd(fmin=8, fmax=14, picks=channels).plot(show=False)
        plt.title('Power Spectral Density (8-14 Hz)')
        plt.savefig('psd_c3_c4.png')
        plt.close()
    except ValueError as e:
        print(f"Error plotting PSD: {e}. Check if channels {channels} exist.")

def plot_erd_ers(bci, trial_idx=4):
    """Plot ERD/ERS for C3/C4 during a trial."""
    try:
        label_field = bci['chaninfo'][0, 0]['label'] if hasattr(bci, 'dtype') else bci['chaninfo']['label']
        c3_idx = np.where(np.array(label_field).flatten() == 'C3')[0][0]
        c4_idx = np.where(np.array(label_field).flatten() == 'C4')[0][0]
    except (IndexError, KeyError):
        print("Error: C3 or C4 not found in channel labels.")
        return
    
    eeg_c3 = bci['data'][0, trial_idx][0][c3_idx, :]
    eeg_c4 = bci['data'][0, trial_idx][0][c4_idx, :]
    sfreq = bci['SRATE'][0, 0]
    
    # Compute PSD for inter-trial (-2000 to 0 ms) and feedback (2000 ms to resultind)
    try:
        resultind = bci['TrialData'][0, trial_idx]['resultind'][0, 0]
    except (IndexError, KeyError):
        print(f"Error accessing TrialData.resultind for trial {trial_idx}")
        return
    
    inter_trial = slice(0, 2000)  # -2000 to 0 ms
    feedback = slice(4000, resultind)  # 2000 ms to end of feedback
    
    freqs, psd_c3_inter = welch(eeg_c3[inter_trial], fs=sfreq, nperseg=1024)
    _, psd_c3_feed = welch(eeg_c3[feedback], fs=sfreq, nperseg=1024)
    _, psd_c4_inter = welch(eeg_c4[inter_trial], fs=sfreq, nperseg=1024)
    _, psd_c4_feed = welch(eeg_c4[feedback], fs=sfreq, nperseg=1024)
    
    # Compute ERD: (feedback power - inter-trial power) / inter-trial power
    alpha_idx = (freqs >= 8) & (freqs <= 14)
    erd_c3 = (np.mean(psd_c3_feed[alpha_idx]) - np.mean(psd_c3_inter[alpha_idx])) / np.mean(psd_c3_inter[alpha_idx])
    erd_c4 = (np.mean(psd_c4_feed[alpha_idx]) - np.mean(psd_c4_inter[alpha_idx])) / np.mean(psd_c4_inter[alpha_idx])
    
    # Plot
    plt.figure(figsize=(6, 4))
    plt.bar(['C3', 'C4'], [erd_c3, erd_c4], color=['red', 'blue'])
    plt.ylabel('ERD/ERS (Relative Power Change)')
    plt.title(f'ERD/ERS for Trial {trial_idx + 1}')
    plt.savefig('erd_ers.png')
    plt.close()

def plot_topography(bci, trial_idx=4):
    """Plot topographic map of alpha power."""
    try:
        label_field = bci['chaninfo'][0, 0]['label'] if hasattr(bci, 'dtype') else bci['chaninfo']['label']
        ch_names = label_field.flatten().tolist() if isinstance(label_field, np.ndarray) else list(label_field)
    except (IndexError, KeyError) as e:
        print(f"Error accessing chaninfo.label for topography: {e}")
        return
    
    eeg_data = bci['data'][0, trial_idx][0] * 1e-6  # Convert to V
    sfreq = bci['SRATE'][0, 0]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(eeg_data, info)
    raw.set_montage('standard_1020')
    raw.filter(8, 14)
    
    psd, freqs = mne.time_frequency.psd_welch(raw, fmin=8, fmax=14, n_per_seg=1024)
    alpha_power = psd.mean(axis=1)
    mne.viz.plot_topomap(alpha_power, raw.info, show=False)
    plt.title(f'Alpha Power Topography (Trial {trial_idx + 1})')
    plt.savefig('topography.png')
    plt.close()

def compute_pvc(bci):
    """Compute Percent Valid Correct (PVC)."""
    try:
        results = np.array([bci['TrialData'][0, i]['result'][0, 0] for i in range(bci['TrialData'].shape[1])])
        hits = np.sum(results == 1)
        misses = np.sum(results == 0)
        return hits / (hits + misses) * 100 if (hits + misses) > 0 else 0
    except (IndexError, KeyError) as e:
        print(f"Error computing PVC: {e}")
        return None

def plot_pvc_across_sessions(subject_id='S1'):
    """Plot PVC across sessions for a subject."""
    pvc_list = []
    session_files = sorted(glob.glob(f'{subject_id}_Session_*.mat'))
    if not session_files:
        print(f"No session files found for {subject_id}")
        return
    
    for file in session_files:
        bci = load_bci_data(file)
        pvc = compute_pvc(bci)
        if pvc is not None:
            pvc_list.append(pvc)
    
    if pvc_list:
        plt.figure(figsize=(6, 4))
        plt.plot(range(1, len(pvc_list) + 1), pvc_list, marker='o', color='#1e88e5')
        plt.xlabel('Session')
        plt.ylabel('Percent Valid Correct (%)')
        plt.title(f'PVC Across Sessions for {subject_id}')
        plt.grid(True)
        plt.savefig('pvc_sessions.png')
        plt.close()
    else:
        print("No valid PVC data to plot.")

In [9]:
# 3. MFDFA Analysis
def perform_mfdfa(bci, channel='C3', trial_idx=4):
    """Perform MFDFA on a single channel's EEG data."""
    try:
        label_field = bci['chaninfo'][0, 0]['label'] if hasattr(bci, 'dtype') else bci['chaninfo']['label']
        ch_idx = np.where(np.array(label_field).flatten() == channel)[0][0]
    except (IndexError, KeyError):
        print(f"Error: Channel {channel} not found in channel labels.")
        return None, None, None
    
    eeg_data = bci['data'][0, trial_idx][0][ch_idx, :]
    
    # MFDFA parameters
    scales = np.arange(16, 2048, 32)  # Scales for detrended fluctuation
    q = np.linspace(-5, 5, 101)  # q-orders for multifractal analysis
    
    # Compute MFDFA
    lag, dfa = MFDFA(eeg_data, scales, q)
    
    # Calculate Hurst exponent and singularity spectrum
    h_q = []
    for i in range(len(q)):
        coef = np.polyfit(np.log2(lag), np.log2(dfa[i, :]), 1)
        h_q.append(coef[0])
    
    # Singularity spectrum
    alpha = h_q + q * np.gradient(h_q, q[1] - q[0])
    f_alpha = q * alpha - (h_q * q - np.log2(dfa[:, -1]))
    
    # Plot multifractal spectrum
    plt.figure(figsize=(6, 4))
    plt.plot(alpha, f_alpha, 'b-')
    plt.xlabel('Singularity Strength (α)')
    plt.ylabel('Multifractal Spectrum f(α)')
    plt.title(f'MFDFA Spectrum for {channel} (Trial {trial_idx + 1})')
    plt.grid(True)
    plt.savefig('mfdfa_spectrum.png')
    plt.close()
    
    return alpha, f_alpha, h_q

In [10]:
# Main execution
if __name__ == '__main__':
    # Load sample .mat file
    file_path = 'S1_Session_1.mat'  # Replace with your file path
    try:
        bci = load_bci_data(file_path)
        
        # Inspect BCI structure
        print("BCI type:", type(bci))
        print("BCI shape:", bci.shape if isinstance(bci, np.ndarray) else "N/A")
        print("BCI dtype.names:", bci.dtype.names if hasattr(bci, 'dtype') else "N/A")
        print("BCI keys:", list(bci.keys()) if isinstance(bci, dict) else "Not a dict")
        if hasattr(bci, 'dtype') and bci.dtype.names:
            print("chaninfo content:", bci['chaninfo'])
            if bci['chaninfo'].dtype.names:
                print("chaninfo dtype.names:", bci['chaninfo'].dtype.names)
                print("chaninfo[0, 0]['label'] content:", bci['chaninfo'][0, 0]['label'])
        elif isinstance(bci, dict) and 'chaninfo' in bci:
            print("chaninfo content:", bci['chaninfo'])
            if isinstance(bci['chaninfo'], dict):
                print("chaninfo keys:", list(bci['chaninfo'].keys()))
            elif bci['chaninfo'].dtype.names:
                print("chaninfo dtype.names:", bci['chaninfo'].dtype.names)
        
        # Create MNE Raw object for trial 5
        raw = create_mne_raw(bci, trial_idx=4)
        
        # Generate visualizations
        plot_time_series(raw, channels=['C3', 'C4'])
        plot_psd(raw, channels=['C3', 'C4'])
        plot_erd_ers(bci, trial_idx=4)
        plot_topography(bci, trial_idx=4)
        plot_pvc_across_sessions(subject_id='S1')
        
        # run mfdfa from prev funtion
        alpha, f_alpha, h_q = perform_mfdfa(bci, channel='C3', trial_idx=4)
        if alpha is not None:
            print(f'Multifractal spectrum width: {max(alpha) - min(alpha):.2f}')
    except Exception as e:
        print(f"Error in main execution: {e}")

BCI type: <class 'numpy.void'>
BCI shape: N/A
BCI dtype.names: ('data', 'time', 'positionx', 'positiony', 'SRATE', 'TrialData', 'metadata', 'chaninfo')
BCI keys: Not a dict
chaninfo content: [[(array([[array(['FP1'], dtype='<U3'), array(['FPZ'], dtype='<U3'),
          array(['FP2'], dtype='<U3'), array(['AF3'], dtype='<U3'),
          array(['AF4'], dtype='<U3'), array(['F7'], dtype='<U2'),
          array(['F5'], dtype='<U2'), array(['F3'], dtype='<U2'),
          array(['F1'], dtype='<U2'), array(['FZ'], dtype='<U2'),
          array(['F2'], dtype='<U2'), array(['F4'], dtype='<U2'),
          array(['F6'], dtype='<U2'), array(['F8'], dtype='<U2'),
          array(['FT7'], dtype='<U3'), array(['FC5'], dtype='<U3'),
          array(['FC3'], dtype='<U3'), array(['FC1'], dtype='<U3'),
          array(['FCZ'], dtype='<U3'), array(['FC2'], dtype='<U3'),
          array(['FC4'], dtype='<U3'), array(['FC6'], dtype='<U3'),
          array(['FT8'], dtype='<U3'), array(['T7'], dtype='<U2'),
  